# LangGraph Swarm Agents

## 학습 목표
- LangGraph + Swarm의 개념과 장점 이해
- 멀티 에이전트 패턴 비교 (Tool-calling, Supervisor, Swarm)
- Swarm 핵심 API 학습 및 실습
- 에이전트 간 핸드오프(handoff) 구현


## LangGraph + Swarm이란?
- [LangGraph Swarm 공식 문서](https://reference.langchain.com/python/langgraph/swarm/)


### Swarm의 개념
**Swarm**은 여러 에이전트가 협업하여 복잡한 작업을 수행하는 멀티 에이전트 시스템입니다.

### 핵심 특징
1. **경량화된 멀티 에이전트 오케스트레이션**: OpenAI의 Swarm 패턴에서 영감을 받음
2. **동적 핸드오프**: 에이전트 간 제어권을 동적으로 전환
3. **컨텍스트 유지**: 대화 상태와 히스토리를 유지하면서 에이전트 전환
4. **도구 기반 전환**: 특수한 핸드오프 도구를 통해 에이전트 간 이동


## 단일 에이전트 vs 멀티 에이전트


### 단일 에이전트 (Single Agent)
```
사용자 → Agent → 도구들 → 응답
```
**장점:**
- 간단한 구조
- 빠른 응답
- 관리 용이

**단점:**
- 복잡한 작업 처리 어려움
- 전문화된 작업 처리 제한
- 컨텍스트 관리 복잡


### 멀티 에이전트 (Multi-Agent)
```
사용자 → 조정자 → Agent1 (전문가1)
              ↓
              → Agent2 (전문가2)
              ↓
              → Agent3 (전문가3)
```
**장점:**
- 각 에이전트가 특정 도메인 전문화
- 복잡한 워크플로우 처리
- 병렬 처리 가능

**단점:**
- 복잡한 오케스트레이션 필요
- 디버깅 어려움
- 오버헤드 증가


## 멀티 에이전트 패턴 비교

### Tool-Calling 패턴
- **구조**: 단일 에이전트가 여러 도구를 호출
- **제어**: 중앙 집중식
- **용도**: 도구 기반 작업 자동화
```python
agent → [tool1, tool2, tool3, ...]
```

### Supervisor 패턴
- **구조**: 슈퍼바이저가 워커 에이전트들을 관리
- **제어**: 슈퍼바이저가 중앙에서 결정
- **용도**: 명확한 계층 구조가 필요한 경우
```python
Supervisor
    ├→ Worker Agent 1
    ├→ Worker Agent 2
    └→ Worker Agent 3
```

### Swarm 패턴
- **구조**: 에이전트 간 동적 핸드오프
- **제어**: 분산형, 각 에이전트가 다음 에이전트 결정
- **용도**: 유연한 대화 흐름, 복잡한 협업
```python
Agent A ←→ Agent B
    ↓         ↓
Agent C ←→ Agent D
```

### 비교표

| 특성 | Tool-Calling | Supervisor | Swarm |
|------|--------------|------------|-------|
| **복잡도** | 낮음 | 중간 | 중간 |
| **유연성** | 낮음 | 중간 | 높음 |
| **제어 방식** | 중앙집중 | 계층적 | 분산형 |
| **적합 사례** | 단순 작업 | 명확한 워크플로우 | 동적 대화 |
| **확장성** | 제한적 | 좋음 | 매우 좋음 |


## Swarm 핵심 API

### 주요 컴포넌트

#### 1. `create_swarm()`
멀티 에이전트 Swarm을 생성하는 메인 함수

```python
create_swarm(
    agents: list[Pregel],           # 에이전트 리스트
    default_active_agent: str,       # 기본 활성 에이전트 이름
    state_schema: StateSchemaType,   # 상태 스키마 (기본: SwarmState)
    context_schema: type[Any] | None # 컨텍스트 스키마
) -> StateGraph
```

#### 2. `SwarmState`
Swarm의 상태를 관리하는 스키마
- `MessagesState`를 상속
- 메시지 히스토리와 현재 활성 에이전트 정보 포함

#### 3. `create_handoff_tool()`
에이전트 간 제어권 전환 도구 생성

```python
create_handoff_tool(
    agent_name: str,        # 전환할 에이전트 이름
    name: str | None,       # 도구 이름 (옵션)
    description: str | None # 도구 설명 (옵션)
) -> BaseTool
```

#### 4. `add_active_agent_router()`
StateGraph에 활성 에이전트 라우터 추가

```python
add_active_agent_router(
    builder: StateGraph,
    route_to: list[str],           # 라우팅할 에이전트 이름 리스트
    default_active_agent: str      # 기본 활성 에이전트
) -> StateGraph
```

# 실습 예제 

## 실습 1: LLM 초기화


In [ ]:
# # 필요한 패키지 임포트
# from typing import Annotated
# from langchain_openai import ChatOpenAI
# from langchain.agents import create_agent
# from langchain_core.tools import tool
# from langgraph_swarm import create_swarm, create_handoff_tool
# import os

# # OpenAI API 키 설정 (환경 변수 또는 직접 설정)
# # os.environ["OPENAI_API_KEY"] = "your-api-key-here"

# # LLM 초기화
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


In [ ]:
from langchain_ollama.chat_models import ChatOllama 

llm = ChatOllama(
    model="gemma3:4b",
    temperature=0.1,
    top_p=1.0,
    num_predict=256,
    keep_alive="5m"
)

## 실습 2: 에이전트별 전문 도구 정의

각 에이전트가 사용할 전문 도구들을 정의합니다.


### 계산 전담 에이전트 도구

In [ ]:
from langchain_core.tools import tool

@tool
def add_numbers(a: float, b: float) -> float:
    """두 숫자를 더합니다."""
    return a + b

@tool
def subtract_numbers(a: float, b: float) -> float:
    """두 숫자를 뺍니다."""
    return a - b

@tool
def multiply_numbers(a: float, b: float) -> float:
    """두 숫자를 곱합니다."""
    return a * b

@tool
def divide_numbers(a: float, b: float) -> float:
    """두 숫자를 나눕니다."""
    if b == 0:
        return "0으로 나눌 수 없습니다."
    return a / b



In [ ]:
# 계산기 도구 목록
calculator_tools = [add_numbers, subtract_numbers, multiply_numbers, divide_numbers]

print("계산 도구 정의 완료")
for tool in calculator_tools:
    print("=" * 50)
    print(f"Tool 이름: {tool.name}")
    print(f"Tool 설명: {tool.description}")
    print(f"Tool 인풋 파라미터: {tool.args}")
    print(f"Tool return_direct: {tool.return_direct}")

### 텍스트 처리 전담 에이전트 도구

In [ ]:
@tool
def count_words(text: str) -> int:
    """텍스트의 단어 수를 세어 반환합니다."""
    return len(text.split())

@tool
def reverse_text(text: str) -> str:
    """텍스트를 역순으로 뒤집습니다."""
    return text[::-1]

@tool
def to_uppercase(text: str) -> str:
    """텍스트를 대문자로 변환합니다."""
    return text.upper()

@tool
def to_lowercase(text: str) -> str:
    """텍스트를 소문자로 변환합니다."""
    return text.lower()

@tool
def count_characters(text: str) -> int:
    """텍스트의 문자 수를 세어 반환합니다 (공백 포함)."""
    return len(text)


In [ ]:
# 텍스트 처리 도구 목록
text_tools = [count_words, reverse_text, to_uppercase, to_lowercase, count_characters]

print("텍스트 처리 도구 정의 완료")
for tool in text_tools:
    print("=" * 50)
    print(f"Tool 이름: {tool.name}")
    print(f"Tool 설명: {tool.description}")
    print(f"Tool 인풋 파라미터: {tool.args}")
    print(f"Tool return_direct: {tool.return_direct}")

### 검색 전담 에이전트 도구

In [ ]:
# 간단한 데이터베이스 시뮬레이션
knowledge_base = {
    "python": "Python은 1991년 귀도 반 로섬이 만든 인터프리터 프로그래밍 언어입니다.",
    "langgraph": "LangGraph는 LLM 애플리케이션의 복잡한 워크플로우를 구축하기 위한 프레임워크입니다.",
    "swarm": "Swarm은 여러 에이전트가 협업하여 복잡한 작업을 수행하는 멀티 에이전트 시스템입니다.",
    "ai": "인공지능(AI)은 인간의 학습능력, 추론능력, 지각능력 등을 컴퓨터 프로그램으로 실현한 기술입니다.",
    "llm": "대규모 언어 모델(LLM)은 방대한 텍스트 데이터로 학습된 인공지능 모델입니다."
}

@tool
def search_knowledge(query: str) -> str:
    """지식 베이스에서 정보를 검색합니다."""
    query_lower = query.lower()
    for key, value in knowledge_base.items():
        if key in query_lower:
            return f"검색 결과: {value}"
    return f"'{query}'에 대한 정보를 찾을 수 없습니다."

@tool
def list_topics() -> str:
    """지식 베이스에 있는 모든 주제를 나열합니다."""
    topics = ", ".join(knowledge_base.keys())
    return f"사용 가능한 주제: {topics}"

@tool
def add_knowledge(topic: str, content: str) -> str:
    """지식 베이스에 새로운 정보를 추가합니다."""
    knowledge_base[topic.lower()] = content
    return f"'{topic}' 주제가 추가되었습니다."


In [ ]:
# 검색 도구 목록
search_tools = [search_knowledge, list_topics, add_knowledge]

print("검색 도구 정의 완료")
for tool in search_tools:
    print("=" * 50)
    print(f"Tool 이름: {tool.name}")
    print(f"Tool 설명: {tool.description}")
    print(f"Tool 인풋 파라미터: {tool.args}")
    print(f"Tool return_direct: {tool.return_direct}")

## 실습 3: 전문 에이전트 생성 및 Handoff 도구 설정

각 에이전트를 생성하고, 다른 에이전트로 작업을 넘길 수 있는 핸드오프 도구를 추가합니다.


In [ ]:
### 1. Handoff 도구 생성
from langgraph_swarm import create_handoff_tool


# 각 에이전트로 넘기기 위한 핸드오프 도구들 생성
handoff_to_calculator = create_handoff_tool(
    agent_name="calculator_agent",
    name="transfer_to_calculator",
    description="계산이 필요할 때 계산기 에이전트로 작업을 전달합니다. 덧셈, 뺄셈, 곱셈, 나눗셈 등의 수학 연산에 사용합니다."
)

handoff_to_text = create_handoff_tool(
    agent_name="text_agent",
    name="transfer_to_text",
    description="텍스트 처리가 필요할 때 텍스트 에이전트로 작업을 전달합니다. 단어 수 세기, 대소문자 변환, 텍스트 뒤집기 등에 사용합니다."
)

handoff_to_search = create_handoff_tool(
    agent_name="search_agent",
    name="transfer_to_search",
    description="정보 검색이 필요할 때 검색 에이전트로 작업을 전달합니다. 지식 베이스에서 정보를 찾거나 주제를 나열할 때 사용합니다."
)

handoff_to_coordinator = create_handoff_tool(
    agent_name="coordinator_agent",
    name="transfer_to_coordinator",
    description="작업을 완료했거나 다른 에이전트가 필요할 때 조정자 에이전트로 돌아갑니다."
)

print("Handoff 도구 생성 완료")
print("   - transfer_to_calculator: 계산 작업 전달")
print("   - transfer_to_text: 텍스트 처리 작업 전달")
print("   - transfer_to_search: 검색 작업 전달")
print("   - transfer_to_coordinator: 조정자로 복귀")


In [ ]:
### 2. 전문 에이전트 생성
from langchain.agents import create_agent

# 1. 조정자 에이전트 (Coordinator)
# - 사용자의 요청을 받고 적절한 전문 에이전트에게 작업을 분배
coordinator_agent = create_agent(
    llm,
    tools=[handoff_to_calculator, handoff_to_text, handoff_to_search],
    state_modifier="""당신은 조정자 에이전트입니다.
사용자의 요청을 분석하고 적절한 전문 에이전트에게 작업을 전달하는 역할을 합니다.

- 계산이 필요하면 → calculator_agent로 전달
- 텍스트 처리가 필요하면 → text_agent로 전달  
- 정보 검색이 필요하면 → search_agent로 전달

각 에이전트의 역할을 파악하고 가장 적합한 에이전트를 선택하세요."""
)
coordinator_agent.name = "coordinator_agent"

# 2. 계산기 에이전트 (Calculator)
# - 수학 연산 전담
calculator_agent = create_agent(
    llm,
    tools=calculator_tools + [handoff_to_coordinator],
    state_modifier="""당신은 계산 전문 에이전트입니다.
수학적 계산(덧셈, 뺄셈, 곱셈, 나눗셈)을 수행하는 것이 당신의 역할입니다.

계산을 완료하면 반드시 transfer_to_coordinator를 사용하여 조정자에게 결과를 보고하세요."""
)
calculator_agent.name = "calculator_agent"

# 3. 텍스트 처리 에이전트 (Text Processor)
# - 텍스트 변환 및 분석 전담
text_agent = create_agent(
    llm,
    tools=text_tools + [handoff_to_coordinator],
    state_modifier="""당신은 텍스트 처리 전문 에이전트입니다.
텍스트 관련 작업(단어 수 세기, 대소문자 변환, 텍스트 뒤집기 등)을 수행하는 것이 당신의 역할입니다.

작업을 완료하면 반드시 transfer_to_coordinator를 사용하여 조정자에게 결과를 보고하세요."""
)
text_agent.name = "text_agent"

# 4. 검색 에이전트 (Search Agent)
# - 정보 검색 및 지식 관리 전담
search_agent = create_agent(
    llm,
    tools=search_tools + [handoff_to_coordinator],
    state_modifier="""당신은 검색 전문 에이전트입니다.
지식 베이스에서 정보를 검색하고 관리하는 것이 당신의 역할입니다.

검색을 완료하면 반드시 transfer_to_coordinator를 사용하여 조정자에게 결과를 보고하세요."""
)
search_agent.name = "search_agent"

print("✅ 전문 에이전트 생성 완료")
print("   1. coordinator_agent: 작업 조정 및 분배")
print("   2. calculator_agent: 수학 계산")
print("   3. text_agent: 텍스트 처리")
print("   4. search_agent: 정보 검색")


## 실습 4: Swarm 시스템 구성

생성한 에이전트들을 하나의 Swarm 시스템으로 통합합니다.


In [ ]:
from langgraph_swarm import create_swarm

# Swarm 생성
swarm = create_swarm(
    agents=[
        coordinator_agent,
        calculator_agent,
        text_agent,
        search_agent
    ],
    default_active_agent="coordinator_agent"  # 기본 시작 에이전트
)

# Swarm 컴파일
swarm_graph = swarm.compile()

print("✅ Swarm 시스템 구성 완료")
print("\n📊 시스템 구조:")
print("   coordinator_agent (조정자)")
print("   ├─ calculator_agent (계산)")
print("   ├─ text_agent (텍스트)")
print("   └─ search_agent (검색)")
print("\n💡 조정자가 사용자 요청을 분석하여 적절한 전문 에이전트에게 작업을 분배합니다.")


## 실습 5: Task Flow 테스트

이제 구축한 Swarm 시스템을 다양한 시나리오로 테스트해봅니다.


In [ ]:
# 테스트를 위한 헬퍼 함수
def run_swarm(user_message: str):
    """Swarm에 메시지를 전달하고 결과를 출력하는 헬퍼 함수"""
    print(f"\n{'='*70}")
    print(f"👤 사용자: {user_message}")
    print(f"{'='*70}\n")
    
    result = swarm_graph.invoke({
        "messages": [{"role": "user", "content": user_message}]
    })
    
    # 에이전트 전환 과정 출력
    print("🔄 에이전트 전환 과정:")
    for i, msg in enumerate(result["messages"], 1):
        if hasattr(msg, 'name') and msg.name:
            print(f"   {i}. [{msg.name}]")
        elif msg.type == "ai":
            print(f"   {i}. [AI 응답]")
    
    # 최종 응답 출력
    final_message = result["messages"][-1]
    print(f"\n💬 최종 응답:")
    print(f"   {final_message.content}")
    print(f"\n{'='*70}\n")
    
    return result

print("✅ 헬퍼 함수 준비 완료")


In [ ]:
### 테스트 1: 계산 작업 (Calculator Agent)

run_swarm("123과 456을 더해줘")


In [ ]:
### 테스트 2: 텍스트 처리 작업 (Text Agent)

run_swarm("'Hello World'라는 텍스트를 대문자로 변환하고 단어 수를 세어줘")


In [ ]:
### 테스트 3: 검색 작업 (Search Agent)

run_swarm("Python에 대해 알려줘")


In [ ]:
### 테스트 4: 복합 작업 (Multiple Agents)

# 여러 에이전트를 순차적으로 사용하는 복합 작업
run_swarm("100에서 45를 뺀 결과를 알려주고, 그 숫자를 텍스트로 변환해서 문자 수를 세어줘")


## 실습 6: 고급 Task Flow - 데이터 분석 파이프라인

더 복잡한 시나리오를 위한 데이터 분석 파이프라인을 구축해봅니다.


In [ ]:
### 1. 데이터 분석 도구 정의

# 간단한 데이터 저장소
data_store = {}

@tool
def store_data(key: str, value: str) -> str:
    """데이터를 저장합니다."""
    data_store[key] = value
    return f"'{key}' 키로 데이터가 저장되었습니다."

@tool
def retrieve_data(key: str) -> str:
    """저장된 데이터를 조회합니다."""
    if key in data_store:
        return f"'{key}' 데이터: {data_store[key]}"
    return f"'{key}' 키를 찾을 수 없습니다."

@tool
def list_all_data() -> str:
    """저장된 모든 데이터를 나열합니다."""
    if not data_store:
        return "저장된 데이터가 없습니다."
    return "저장된 데이터:\n" + "\n".join([f"- {k}: {v}" for k, v in data_store.items()])

@tool
def calculate_average(numbers_str: str) -> str:
    """쉼표로 구분된 숫자들의 평균을 계산합니다. 예: '10,20,30'"""
    try:
        numbers = [float(n.strip()) for n in numbers_str.split(',')]
        avg = sum(numbers) / len(numbers)
        return f"평균: {avg}"
    except:
        return "올바른 형식이 아닙니다. 예: '10,20,30'"

data_analysis_tools = [store_data, retrieve_data, list_all_data, calculate_average]

print("✅ 데이터 분석 도구 정의 완료")


In [ ]:
### 2. 데이터 분석 에이전트 생성 및 확장 Swarm 구성

# 데이터 분석 에이전트로 핸드오프 도구
handoff_to_data_analyst = create_handoff_tool(
    agent_name="data_analyst_agent",
    name="transfer_to_data_analyst",
    description="데이터 저장, 조회, 평균 계산 등 데이터 분석이 필요할 때 데이터 분석 에이전트로 작업을 전달합니다."
)

# 기존 조정자 에이전트를 새로운 핸드오프 도구로 업데이트
advanced_coordinator_agent = create_agent(
    llm,
    tools=[handoff_to_calculator, handoff_to_text, handoff_to_search, handoff_to_data_analyst],
    state_modifier="""당신은 조정자 에이전트입니다.
사용자의 요청을 분석하고 적절한 전문 에이전트에게 작업을 전달하는 역할을 합니다.

- 계산이 필요하면 → calculator_agent로 전달
- 텍스트 처리가 필요하면 → text_agent로 전달  
- 정보 검색이 필요하면 → search_agent로 전달
- 데이터 저장/조회/분석이 필요하면 → data_analyst_agent로 전달

각 에이전트의 역할을 파악하고 가장 적합한 에이전트를 선택하세요."""
)
advanced_coordinator_agent.name = "coordinator_agent"

# 데이터 분석 에이전트
data_analyst_agent = create_agent(
    llm,
    tools=data_analysis_tools + [handoff_to_coordinator, handoff_to_calculator],
    state_modifier="""당신은 데이터 분석 전문 에이전트입니다.
데이터 저장, 조회, 평균 계산 등 데이터 관련 작업을 수행합니다.

필요한 경우:
- 복잡한 수학 계산이 필요하면 calculator_agent로 전달
- 작업을 완료하면 coordinator로 복귀

작업을 완료하면 반드시 transfer_to_coordinator를 사용하여 조정자에게 결과를 보고하세요."""
)
data_analyst_agent.name = "data_analyst_agent"

# 확장된 Swarm 생성
advanced_swarm = create_swarm(
    agents=[
        advanced_coordinator_agent,
        calculator_agent,
        text_agent,
        search_agent,
        data_analyst_agent
    ],
    default_active_agent="coordinator_agent"
)

advanced_swarm_graph = advanced_swarm.compile()

print("✅ 확장 Swarm 시스템 구성 완료")
print("\n📊 확장된 시스템 구조:")
print("   coordinator_agent (조정자)")
print("   ├─ calculator_agent (계산)")
print("   ├─ text_agent (텍스트)")
print("   ├─ search_agent (검색)")
print("   └─ data_analyst_agent (데이터 분석)")
print("      └─ calculator_agent로 연결 가능")
print("\n💡 데이터 분석 에이전트가 추가되어 더 복잡한 워크플로우를 처리할 수 있습니다.")


In [ ]:
### 3. 고급 헬퍼 함수 및 테스트

# 고급 Swarm을 위한 헬퍼 함수
def run_advanced_swarm(user_message: str):
    """고급 Swarm에 메시지를 전달하고 결과를 출력하는 헬퍼 함수"""
    print(f"\n{'='*70}")
    print(f"👤 사용자: {user_message}")
    print(f"{'='*70}\n")
    
    result = advanced_swarm_graph.invoke({
        "messages": [{"role": "user", "content": user_message}]
    })
    
    # 에이전트 전환 과정 출력
    print("🔄 에이전트 전환 과정:")
    agent_sequence = []
    for msg in result["messages"]:
        if hasattr(msg, 'name') and msg.name:
            if not agent_sequence or agent_sequence[-1] != msg.name:
                agent_sequence.append(msg.name)
    
    print(f"   {' → '.join(agent_sequence)}")
    
    # 최종 응답 출력
    final_message = result["messages"][-1]
    print(f"\n💬 최종 응답:")
    print(f"   {final_message.content}")
    print(f"\n{'='*70}\n")
    
    return result

print("✅ 고급 헬퍼 함수 준비 완료")


In [ ]:
### 테스트 5: 데이터 저장 및 조회

run_advanced_swarm("'sales_q1'이라는 키로 '150000' 값을 저장해줘")


In [ ]:
### 테스트 6: 복잡한 데이터 분석 파이프라인

# 여러 단계의 작업이 필요한 복잡한 시나리오
run_advanced_swarm("다음 숫자들의 평균을 계산해줘: 85, 92, 78, 95, 88. 그리고 그 결과를 'exam_average'로 저장해줘.")


In [ ]:
### 테스트 7: 저장된 모든 데이터 확인

run_advanced_swarm("지금까지 저장한 모든 데이터를 보여줘")


## 실습 요약: 핵심 개념

이 실습을 통해 학습한 핵심 개념들을 정리합니다.


### 1️⃣ 에이전트에 역할 부여

```python
# 각 에이전트는 state_modifier를 통해 명확한 역할을 부여받음
agent = create_agent(
    llm,
    tools=[...],
    state_modifier="당신은 [역할] 전문 에이전트입니다. [구체적인 역할 설명]"
)
```

**장점:**
- 각 에이전트가 특정 도메인에 전문화
- 명확한 책임 분리
- 더 정확하고 일관된 결과

### 2️⃣ 에이전트에 도구(Tools) 연결

```python
# @tool 데코레이터로 함수를 도구로 변환
@tool
def my_function(param: str) -> str:
    """도구 설명 (LLM이 이를 보고 언제 사용할지 결정)"""
    return result

# 에이전트 생성 시 도구 연결
agent = create_agent(llm, tools=[tool1, tool2, tool3])
```

**핵심:**
- 도구의 docstring이 매우 중요 (LLM이 이를 보고 판단)
- 각 에이전트는 필요한 도구만 가져야 함

### 3️⃣ Handoff를 통한 에이전트 전환

```python
# 다른 에이전트로 작업을 넘기는 도구 생성
handoff_tool = create_handoff_tool(
    agent_name="target_agent",
    name="transfer_to_target",
    description="[언제 이 에이전트로 전환해야 하는지 설명]"
)

# 조정자 에이전트에 핸드오프 도구 추가
coordinator = create_agent(
    llm,
    tools=[handoff_to_agent1, handoff_to_agent2, ...]
)
```

**Task Flow 설계:**
1. **조정자(Coordinator)**: 요청 분석 → 적절한 전문 에이전트 선택
2. **전문 에이전트**: 작업 수행 → 조정자에게 복귀
3. **필요시 연쇄**: 전문 에이전트 → 다른 전문 에이전트 → 조정자

### 4️⃣ Swarm 생성 및 실행

```python
# 모든 에이전트를 하나의 Swarm으로 통합
swarm = create_swarm(
    agents=[coordinator, agent1, agent2, agent3],
    default_active_agent="coordinator"
)

# 실행
graph = swarm.compile()
result = graph.invoke({"messages": [{"role": "user", "content": "..."}]})
```


## 💡 실전 적용 팁

Swarm 시스템을 실제 프로젝트에 적용할 때 고려할 사항들입니다.


### 🎯 에이전트 설계 가이드

1. **단일 책임 원칙**
   - 각 에이전트는 하나의 명확한 역할만 수행
   - 너무 많은 도구를 한 에이전트에 집중시키지 않기

2. **명확한 역할 정의**
   - state_modifier에서 역할을 구체적으로 설명
   - 언제 다른 에이전트로 전환해야 하는지 명시

3. **효율적인 Handoff 설계**
   - 핸드오프 도구의 description을 명확하게 작성
   - 불필요한 에이전트 전환 최소화

### 🔍 디버깅 팁

```python
# 에이전트 전환 과정을 로깅하여 문제 파악
for msg in result["messages"]:
    if hasattr(msg, 'name'):
        print(f"Agent: {msg.name}")
        if hasattr(msg, 'tool_calls'):
            print(f"Tool calls: {msg.tool_calls}")
```

### ⚡ 성능 최적화

1. **적절한 모델 선택**
   - 간단한 작업: `gpt-4o-mini` (빠르고 저렴)
   - 복잡한 추론: `gpt-4` (높은 정확도)

2. **불필요한 핸드오프 감소**
   - 자주 함께 사용되는 도구는 같은 에이전트에 배치
   - 조정자를 거치지 않고 직접 연결 (예: data_analyst → calculator)

3. **컨텍스트 관리**
   - 각 에이전트의 프롬프트를 간결하게 유지
   - 필요한 정보만 state에 저장

### 🌟 실제 활용 사례

1. **고객 서비스 챗봇**
   - 조정자: 고객 의도 파악
   - FAQ 에이전트: 자주 묻는 질문 처리
   - 주문 에이전트: 주문 조회/변경
   - 에스컬레이션 에이전트: 복잡한 문제를 인간 상담원에게 전달

2. **데이터 분석 플랫폼**
   - 데이터 수집 에이전트
   - 전처리 에이전트
   - 분석 에이전트
   - 시각화 에이전트
   - 리포트 생성 에이전트

3. **코드 리뷰 시스템**
   - 코드 분석 에이전트
   - 보안 검사 에이전트
   - 성능 분석 에이전트
   - 문서화 에이전트


## 📝 연습 문제

스스로 구현해보며 학습 내용을 복습해봅시다.


### 연습 1: 날씨 에이전트 추가 ⛅

다음 기능을 가진 날씨 에이전트를 추가해보세요:

1. **도구 작성**
   ```python
   @tool
   def get_weather(city: str) -> str:
       """지정된 도시의 날씨를 조회합니다."""
       # 간단한 시뮬레이션
       weather_data = {
           "서울": "맑음, 15°C",
           "부산": "흐림, 18°C",
           "제주": "비, 20°C"
       }
       return weather_data.get(city, f"{city}의 날씨 정보를 찾을 수 없습니다.")
   ```

2. **에이전트 생성**
   - 날씨 조회 기능을 가진 에이전트 생성
   - 조정자로 복귀하는 핸드오프 도구 추가

3. **Swarm에 통합**
   - 기존 Swarm에 날씨 에이전트 추가
   - 조정자가 날씨 에이전트로 전환할 수 있도록 핸드오프 도구 추가

4. **테스트**
   - "서울의 날씨를 알려줘" 같은 요청으로 테스트

---

### 연습 2: 멀티스텝 워크플로우 🔄

다음 워크플로우를 처리할 수 있는 시스템을 구현해보세요:

**시나리오**: 쇼핑몰 주문 처리 시스템

1. **재고 확인 에이전트**
   - `check_inventory(product: str) -> int`: 재고 수량 반환

2. **가격 계산 에이전트**
   - `calculate_price(product: str, quantity: int) -> float`: 가격 계산
   - `apply_discount(price: float, discount_rate: float) -> float`: 할인 적용

3. **주문 처리 에이전트**
   - `create_order(product: str, quantity: int, price: float) -> str`: 주문 생성

**요구사항**:
- 각 에이전트가 순차적으로 작업 수행
- 조정자가 전체 프로세스 관리
- "노트북 3개 주문하고 10% 할인 적용해줘" 같은 요청 처리

---

### 연습 3: 에러 처리 에이전트 🛠️

에러 상황을 처리하는 에이전트를 추가해보세요:

1. **도구 작성**
   ```python
   @tool
   def validate_input(input_text: str) -> str:
       """입력값의 유효성을 검사합니다."""
       # 구현...
   
   @tool
   def handle_error(error_type: str) -> str:
       """에러를 처리하고 사용자 친화적인 메시지를 반환합니다."""
       # 구현...
   ```

2. **에이전트 역할**
   - 다른 에이전트에서 발생한 에러 처리
   - 사용자에게 명확한 에러 메시지 제공
   - 가능한 경우 대안 제시

---

### 💪 도전 과제: 완전한 애플리케이션 구축

다음 중 하나를 선택하여 완전한 멀티 에이전트 시스템을 구축해보세요:

#### 옵션 1: 여행 계획 도우미 ✈️
- 목적지 추천 에이전트
- 항공권 검색 에이전트
- 숙박 예약 에이전트
- 일정 관리 에이전트

#### 옵션 2: 학습 도우미 📚
- 질문 분석 에이전트
- 개념 설명 에이전트
- 문제 출제 에이전트
- 학습 진도 관리 에이전트

#### 옵션 3: 콘텐츠 제작 스튜디오 🎨
- 아이디어 브레인스토밍 에이전트
- 초안 작성 에이전트
- 편집 및 교정 에이전트
- SEO 최적화 에이전트


## 📚 학습 정리 및 다음 단계

### ✅ 오늘 학습한 내용

1. **Swarm 개념과 패턴**
   - Tool-calling, Supervisor, Swarm 패턴 비교
   - 각 패턴의 장단점과 적용 사례

2. **에이전트 역할 부여**
   - `state_modifier`를 통한 명확한 역할 정의
   - 전문화된 에이전트의 장점

3. **도구(Tools) 연결**
   - `@tool` 데코레이터 사용법
   - 도구의 docstring 중요성
   - 각 에이전트에 적절한 도구 배치

4. **Handoff 메커니즘**
   - `create_handoff_tool()`로 에이전트 간 전환
   - 동적 task flow 설계
   - 조정자 패턴 구현

5. **실전 예제**
   - 계산기, 텍스트 처리, 검색, 데이터 분석 에이전트
   - 복합 작업 처리 파이프라인
   - 에이전트 간 협업 워크플로우

### 🎓 핵심 takeaway

- **적절한 추상화**: 각 에이전트는 하나의 명확한 책임만 가져야 함
- **유연한 협업**: Handoff를 통해 동적으로 작업을 분배
- **확장 가능성**: 새로운 에이전트를 쉽게 추가할 수 있는 구조
- **명확한 역할**: state_modifier와 tool description이 시스템의 핵심

### 🚀 다음 단계

1. **고급 기능 탐구**
   - 상태(State) 커스터마이징
   - 컨텍스트(Context) 스키마 활용
   - 에이전트 간 데이터 공유 전략

2. **프로덕션 준비**
   - 에러 처리 및 복구 메커니즘
   - 로깅 및 모니터링
   - 성능 최적화

3. **실제 프로젝트 적용**
   - 비즈니스 요구사항에 맞는 에이전트 설계
   - 확장 가능한 아키텍처 구축
   - 지속적인 개선 및 최적화

### 📖 추가 학습 자료

- [LangGraph 공식 문서](https://python.langchain.com/docs/langgraph)
- [LangGraph Swarm 레퍼런스](https://reference.langchain.com/python/langgraph/swarm/)
- [OpenAI Swarm (영감을 받은 원본)](https://github.com/openai/swarm)

### 🤝 피드백 및 질문

학습 중 궁금한 점이나 개선 사항이 있다면 언제든지 공유해주세요!
